In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# ==================== Downsampler ====================

class Downsampler(nn.Module):
    def __init__(self, factor):
        super(Downsampler, self).__init__()
        support = 2
        kernel_width = 4 * factor + 1
        self.kernel = self._generate_lanczos_kernel(factor, kernel_width, support)
        self.downsampler_ = self._create_downsampler(factor)
        pad = self._calculate_padding(factor)
        self.padding = nn.ReplicationPad2d(pad)
    
    def _generate_lanczos_kernel(self, factor, kernel_width, support):
        kernel = np.zeros([kernel_width, kernel_width])
        center = (kernel_width + 1) / 2.0
        for i in range(1, kernel.shape[0] + 1):
            for j in range(1, kernel.shape[1] + 1):
                di = abs(i - center) / factor
                dj = abs(j - center) / factor
                val = self._lanczos_value(di, support) * self._lanczos_value(dj, support)
                kernel[i - 1, j - 1] = val
        kernel /= kernel.sum()
        return kernel
    
    @staticmethod
    def _lanczos_value(distance, support):
        if distance == 0:
            return 1.0
        pi_dist = np.pi * distance
        pi_dist_support = pi_dist / support
        return support * np.sin(pi_dist) * np.sin(pi_dist_support) / (pi_dist * pi_dist)
    
    def _create_downsampler(self, factor):
        kernel_size = self.kernel.shape[0]
        downsampler = nn.Conv2d(1, 1, kernel_size=kernel_size, stride=factor, padding=0, bias=True)
        downsampler.weight.data.zero_()
        downsampler.bias.data.zero_()
        kernel_torch = torch.from_numpy(self.kernel).float()
        downsampler.weight.data[0, 0] = kernel_torch
        return downsampler
    
    def _calculate_padding(self, factor):
        kernel_size = self.kernel.shape[0]
        if kernel_size % 2 == 1:
            pad = (kernel_size - 1) // 2
        else:
            pad = (kernel_size - factor) // 2
        return pad
    
    def forward(self, input):
        x = self.padding(input)
        return self.downsampler_(x)

# ==================== Attention Modules ====================

class LightweightAttention(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.channel_attention = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, max(channels // reduction, 1), 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(max(channels // reduction, 1), channels, 1, bias=False),
            nn.Sigmoid()
        )
        self.spatial_attention = nn.Sequential(
            nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        ca = self.channel_attention(x)
        x = x * ca
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        sa_input = torch.cat([avg_out, max_out], dim=1)
        sa = self.spatial_attention(sa_input)
        x = x * sa
        return x

class AttentionGate(nn.Module):
    def __init__(self, skip_channels, decoder_channels, intermediate_channels=None):
        super().__init__()
        if intermediate_channels is None:
            intermediate_channels = max(skip_channels // 2, 1)
        self.W_g = nn.Sequential(
            nn.Conv2d(decoder_channels, intermediate_channels, 1, bias=True),
            nn.BatchNorm2d(intermediate_channels)
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(skip_channels, intermediate_channels, 1, bias=True),
            nn.BatchNorm2d(intermediate_channels)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(intermediate_channels, 1, 1, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, skip_connection, decoder_feature):
        g1 = self.W_g(decoder_feature)
        x1 = self.W_x(skip_connection)
        if g1.shape[2:] != x1.shape[2:]:
            g1 = F.interpolate(g1, size=x1.shape[2:], mode='bilinear', align_corners=False)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return skip_connection * psi

class BottleneckSelfAttention(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.query_conv = nn.Conv2d(channels, max(channels // reduction, 1), 1)
        self.key_conv = nn.Conv2d(channels, max(channels // reduction, 1), 1)
        self.value_conv = nn.Conv2d(channels, channels, 1)
        self.softmax = nn.Softmax(dim=-1)
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        b, c, h, w = x.size()
        proj_query = self.query_conv(x).view(b, -1, h*w).permute(0, 2, 1)
        proj_key   = self.key_conv(x).view(b, -1, h*w)
        proj_value = self.value_conv(x).view(b, -1, h*w)
        energy = torch.bmm(proj_query, proj_key)
        attention = self.softmax(energy)
        out = torch.bmm(proj_value, attention.permute(0, 2, 1))
        out = out.view(b, c, h, w)
        out = self.gamma * out + x
        return out

# ==================== Fully Expanded UNet with Attention ====================
class UNetWithAttention(nn.Module):
    def __init__(self, input_depth, output_depth, num_channels_down, num_channels_up, num_channels_skip):
        super().__init__()

        self.num_channels_down = [128,128,128,128,128]
        self.num_channels_up   = [128,128,128,128,128]
        self.num_channels_skip = [4,4,4,4,4]
        self.need_sigmoid = True

        n_scales = len(self.num_channels_down)

        # ----- Encoder -----
        self.down_blocks = nn.ModuleList()
        self.skip_blocks = nn.ModuleList()
        in_ch = input_depth
        for i in range(n_scales):
            down = nn.Sequential(
                nn.Conv2d(in_ch, self.num_channels_down[i], 3, stride=2, padding=1, bias=True),
                nn.BatchNorm2d(self.num_channels_down[i]),
                nn.LeakyReLU(0.1, inplace=True),
                nn.Conv2d(self.num_channels_down[i], self.num_channels_down[i], 3, padding=1, bias=True),
                nn.BatchNorm2d(self.num_channels_down[i]),
                nn.LeakyReLU(0.1, inplace=True),
                LightweightAttention(self.num_channels_down[i])
            )
            self.down_blocks.append(down)

            if self.num_channels_skip[i] > 0:
                skip = nn.Sequential(
                    nn.Conv2d(in_ch, self.num_channels_skip[i], 1, bias=True),
                    nn.BatchNorm2d(self.num_channels_skip[i]),
                    nn.LeakyReLU(0.1, inplace=True),
                    LightweightAttention(self.num_channels_skip[i])
                )
            else:
                skip = None
            self.skip_blocks.append(skip)
            in_ch = self.num_channels_down[i]

        # ----- Bottleneck -----
        self.bottleneck_attention = BottleneckSelfAttention(self.num_channels_down[-1])

        # ----- Decoder -----
        self.up_blocks = nn.ModuleList()
        self.skip_attentions = nn.ModuleList()
        for i in reversed(range(n_scales)):
            up_in_ch = self.num_channels_down[i] if i==n_scales-1 else self.num_channels_up[i+1]
            concat_ch = up_in_ch + self.num_channels_skip[i]

            if self.num_channels_skip[i] > 0:
                self.skip_attentions.append(
                    AttentionGate(skip_channels=self.num_channels_skip[i], decoder_channels=up_in_ch)
                )
            else:
                self.skip_attentions.append(None)

            up = nn.Sequential(
                nn.Conv2d(concat_ch, self.num_channels_up[i], 3, padding=1, bias=True),
                nn.BatchNorm2d(self.num_channels_up[i]),
                nn.LeakyReLU(0.1, inplace=True),
                nn.Conv2d(self.num_channels_up[i], self.num_channels_up[i], 1, bias=True),
                nn.BatchNorm2d(self.num_channels_up[i]),
                nn.LeakyReLU(0.1, inplace=True),
                LightweightAttention(self.num_channels_up[i])
            )
            self.up_blocks.append(up)

        # ----- Output -----
        self.final_conv = nn.Conv2d(self.num_channels_up[0], output_depth, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        skips = []
        out = x
        # Encoder
        for down, skip in zip(self.down_blocks, self.skip_blocks):
            if skip is not None:
                skips.append(skip(out))
            else:
                skips.append(None)
            out = down(out)

        # Bottleneck
        out = self.bottleneck_attention(out)

        # Decoder
        for up, skip_att, skip_feat in zip(self.up_blocks, self.skip_attentions, reversed(skips)):
            out = F.interpolate(out, scale_factor=2, mode='nearest')
            if skip_feat is not None:
                if skip_att is not None:
                    skip_feat = skip_att(skip_feat, out)
                out = torch.cat([skip_feat, out], dim=1)
            out = up(out)

        out = self.final_conv(out)
        if self.need_sigmoid:
            out = self.sigmoid(out)
        return out
